In [8]:
import pandas as pd
import numpy as np

### 1. 🛡️ `pd.to_numeric()`: El Rey de los Números

Uso exclusivo para convertir series a enteros (`int`) o decimales (`float`). Es la herramienta principal para limpiar columnas de precios o stock.

| Parámetro | Opciones / Uso | ¿Qué hace? |
| --- | --- | --- |
| `arg` | `Serie` o lista | La columna que vas a convertir. |
| **`errors`** | `'raise'` (default) | Rompe el código si hay un error. (¡Evitar en pipelines!) |
|  | **`'coerce'`** | **Modo Pro.** Convierte los errores en `NaN`. El pipeline sigue vivo. |
|  | `'ignore'` | Ignora el error y devuelve la serie original sin cambios. |
| `downcast` | `'integer'`, `'float'` | Reduce el tamaño en memoria (ej: pasa de `float64` a `float32`). |

**Ejemplo Práctico:**

In [ ]:
# Simulamos datos sucios extraídos de una tienda de hardware
df = pd.DataFrame({
    'producto': ['RTX 4060', 'Ryzen 5', 'Gabinete ATX'],
    'precio_ars': ['450000', 'Sin Stock', '85000'],
    'stock': ['15', '0', 'Consultar']
})

# Casteo robusto: los textos se vuelven NaN, los números se convierten a float/int
df['precio_ars'] = pd.to_numeric(df['precio_ars'], errors='coerce')
# Nota: fillna(0) reemplaza el NaN por 0 antes de forzar el entero.
df['stock'] = pd.to_numeric(df['stock'], errors='coerce').fillna(0).astype(int) 
df

,producto,precio_ars,stock
0,RTX 4060,450000.0,15
1,Ryzen 5,NaN,0
2,Gabinete ATX,85000.0,0


---

### 2. ⏱️ `pd.to_datetime()`: El Maestro del Tiempo

Imprescindible para auditar cuándo se extrajo un dato o estandarizar fechas de publicación.

| Parámetro | Opciones / Uso | ¿Qué hace? |
| --- | --- | --- |
| `arg` | `Serie` o lista | La columna con las fechas en texto. |
| **`errors`** | `'raise'`, `'coerce'`, `'ignore'` | Funciona igual que en `to_numeric`. Usa `'coerce'`. |
| `format` | Ej: `'%d/%m/%Y'` | Acelera la conversión indicándole exactamente el formato esperado. |
| `dayfirst` | `True` / `False` | Útil en Argentina para fechas ambiguas (ej: `02/03/2026` -> 2 de marzo, no 3 de febrero). |

**Ejemplo Práctico:**

In [10]:
df['fecha_extraccion'] = ['22/02/2026', '2026-02-22', 'Ayer']

# Forzamos el formato día/mes/año, lo que no cuadra se vuelve NaT (Not a Time)
df['fecha_extraccion'] = pd.to_datetime(
    df['fecha_extraccion'], 
    errors='coerce', 
    dayfirst=True
)

df

,producto,precio_ars,stock,fecha_extraccion
0,RTX 4060,450000.0,15,2026-02-22
1,Ryzen 5,NaN,0,NaT
2,Gabinete ATX,85000.0,0,NaT



---

### 3. 🏷️ `.astype()`: La Herramienta Quirúrgica

Solo debes usar `.astype()` cuando estés **100% seguro** de que no hay basura en la columna, o para tipos de datos específicos como categorías o booleanos.

| Parámetro | Opciones / Uso | ¿Qué hace? |
| --- | --- | --- |
| `dtype` | `'category'`, `bool`, `str` | El tipo de dato destino. |
| `errors` | `'raise'`, `'ignore'` | *Nota:* `.astype()` no tiene la opción `'coerce'`. Por eso es peligroso con números. |

**Ejemplo Práctico (Optimizando Memoria):**

In [11]:
df['tienda'] = ['CompraGamer', 'Venex', 'CompraGamer']

# Ideal para textos que se repiten mucho (diccionario interno)
df['tienda'] = df['tienda'].astype('category') 

df

,producto,precio_ars,stock,fecha_extraccion,tienda
0,RTX 4060,450000.0,15,2026-02-22,CompraGamer
1,Ryzen 5,NaN,0,NaT,Venex
2,Gabinete ATX,85000.0,0,NaT,CompraGamer


In [12]:
# Convertir 1/0 a True/False
df['es_oferta'] = [1, 0, 1]
df['es_oferta'] = df['es_oferta'].astype(bool)

df

,producto,precio_ars,stock,fecha_extraccion,tienda,es_oferta
0,RTX 4060,450000.0,15,2026-02-22,CompraGamer,True
1,Ryzen 5,NaN,0,NaT,Venex,False
2,Gabinete ATX,85000.0,0,NaT,CompraGamer,True


---

### 4. ✨ El Paso Cero: `.str.replace()` (Limpieza Pre-Casteo)

Si intentas usar `pd.to_numeric()` en un valor como `"$ 1.500.000"`, te devolverá `NaN` porque el signo de dólar y los puntos confunden al motor. Siempre limpia el string primero.

In [16]:
# Columna original: ["$ 450.000", "$ 1.200.500", "Sin Stock"]
df2 = pd.DataFrame({
    'precio_crudo' : ["$ 450.000", "$ 1.200.500", "Sin Stock"]
})

# Limpieza
df2['precio_limpio'] = (
    df2['precio_crudo']
    .str.replace('$', '', regex=False)    # Quita el símbolo $
    .str.replace('.', '', regex=False)    # Quita los puntos de miles
    .str.strip()                          # Quita espacios en blanco
)

# Ahora sí, casteamos
df2['precio_final'] = pd.to_numeric(df2['precio_limpio'], errors='coerce')
df2

,precio_crudo,precio_limpio,precio_final
0,$ 450.000,450000,450000.0
1,$ 1.200.500,1200500,1200500.0
2,Sin Stock,Sin Stock,NaN


---

### 5. 🚀 El Truco Moderno: `.convert_dtypes()`

Si usas una versión reciente de Pandas, este método escanea todo tu DataFrame y asigna el "mejor tipo de dato posible" a cada columna de forma automática, soportando valores nulos de forma nativa (como `Int64` con mayúscula).

In [32]:
# Aplica a todo el DataFrame de una vez para inferir tipos lógicos
df = df.convert_dtypes()